# Torch/MLX Equivalent DDPM Matrix

This notebook builds one execution matrix for four DDPM model families across two backends and three beta schedules.

Model families:

1. Base conditional DDPM
2. Guided conditional DDPM (`classifier-free` and `classifier` modes are configurable)
3. Loop-conditioned DDPM
4. Unconditional DDPM

Beta schedules:

- `linear`
- `hash-approach1`
- `hash-approach2`

The Torch jobs call the existing PyTorch training pipelines. The MLX jobs use `ddpm_mlx_equivalent.py`, which mirrors the Torch U-Net structures with MLX `NHWC` tensors.

In [ ]:
from __future__ import annotations

import gc
import json
import math
import sys
from collections.abc import Iterable
from dataclasses import asdict, is_dataclass
from datetime import datetime
from itertools import cycle
from pathlib import Path
from pprint import pprint

import numpy as np
import torch
from torch.utils.data import DataLoader

# -----------------------------------------------------------------------------
# Notebook-level controls
# -----------------------------------------------------------------------------
MODE = "full"  # "smoke" or "full"
# Execute the dedicated job cells below. Running all job cells runs the full matrix.

SELECTED_BACKENDS = ("torch", "mlx")
SELECTED_MODEL_FAMILIES = ("base", "guided", "loop", "unconditional")
SELECTED_BETA_SCHEDULES = ("linear", "hash-approach1", "hash-approach2")
GUIDED_MODES = ("classifier-free", "classifier")

TORCH_DEVICE = "cpu" if MODE == "smoke" else "mps"
MLX_DEVICE = "gpu"  # use "gpu" on Apple Silicon when Metal is available

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "src").exists() and (REPO_ROOT.parent / "src").exists():
    REPO_ROOT = REPO_ROOT.parent

src_path = str(REPO_ROOT / "src")
if src_path not in sys.path:
    sys.path.insert(0, src_path)

DATA_ROOT = REPO_ROOT / "data" / "images"
JSON_ROOT = REPO_ROOT / "output" / "json"
NOTEBOOK_OUTPUT = REPO_ROOT / "output" / f"ddpm_torch_mlx_equiv_{MODE}_{datetime.now().strftime('%Y%m%d_%H%M%S')}"

_SMOKE = dict(
    max_images=16,
    image_size=32,
    channels=3,
    batch_size=2,
    train_steps=2,
    base_channels=4,
    time_dim=8,
    sample_count=1,
    learning_rate=2e-4,
    log_every=1,
)
_FULL = dict(
    max_images=None,
    image_size=64,
    channels=3,
    batch_size=64,
    train_steps=5_000,
    base_channels=64,
    time_dim=256,
    sample_count=4,
    learning_rate=2e-4,
    log_every=50,
)
_cfg = _SMOKE if MODE == "smoke" else _FULL

MAX_IMAGES = _cfg["max_images"]
IMAGE_SIZE = _cfg["image_size"]
CHANNELS = _cfg["channels"]
BATCH_SIZE = _cfg["batch_size"]
TRAIN_STEPS = _cfg["train_steps"]
BASE_CHANNELS = _cfg["base_channels"]
TIME_DIM = _cfg["time_dim"]
SAMPLE_COUNT = _cfg["sample_count"]
LEARNING_RATE = _cfg["learning_rate"]
LOG_EVERY = _cfg["log_every"]

# Use "auto" for every schedule so linear uses the same length as approach 1/2.
TIMESTEPS = "auto"
FIT_MODE = "height-flatten"
EPOCHS = None
SEED = 0
SAVE_PROCESS_TRACES = True
TRACE_SAMPLE_COUNT = 1
SAMPLE_EVERY = 0
CHECKPOINT_EVERY = 0
SAVE_MLX_SAMPLES = True

print("repo:", REPO_ROOT)
print("output:", NOTEBOOK_OUTPUT)
print("data exists:", DATA_ROOT.exists())
print("json exists:", JSON_ROOT.exists())
COUNT_MESSAGE_LIMIT = 1000
if DATA_ROOT.exists():
    _message_count = sum(1 for _ in zip(range(COUNT_MESSAGE_LIMIT + 1), DATA_ROOT.rglob("message.png")))
    _message_count_text = f">={COUNT_MESSAGE_LIMIT}" if _message_count > COUNT_MESSAGE_LIMIT else str(_message_count)
else:
    _message_count_text = "0"
print("message.png count:", _message_count_text)
print("mode:", MODE, "torch:", TORCH_DEVICE, "mlx:", MLX_DEVICE)


In [ ]:
from diffusion_hash_inv.models.conditional_diffusion import (
    ConditionalDiffusionTrainConfig,
    GeneratedImageDataset,
    _ensure_square_batch,
    build_beta_schedule,
    cleanup_torch_resources,
    resolve_train_steps,
    save_image_grid,
    save_sample_artifacts,
    set_seed,
    train_conditional_diffusion,
)
from diffusion_hash_inv.models.guided_conditional_diffusion import (
    GuidedConditionalDiffusionTrainConfig,
    train_guided_conditional_diffusion,
)
from diffusion_hash_inv.models.loop_conditioned_diffusion import (
    LoopConditionedDiffusionTrainConfig,
    LoopConditionedImageDataset,
    train_loop_conditioned_diffusion,
)
from diffusion_hash_inv.models.unconditional_ddpm import (
    UnconditionalDDPMTrainConfig,
    UnconditionalImageDataset,
    train_unconditional_ddpm,
)

from diffusion_hash_inv.models.sample_decoding import write_decode_comparison_summary
MLX_AVAILABLE = False
MLX_IMPORT_ERROR = None
try:
    import mlx.core as mx

    if MLX_DEVICE == "cpu":
        mx.set_default_device(mx.cpu)
    elif MLX_DEVICE == "gpu":
        mx.set_default_device(mx.gpu)
    else:
        raise ValueError(f"Unsupported MLX_DEVICE: {MLX_DEVICE}")

    import mlx.optimizers as mlx_optim
    from diffusion_hash_inv.models import ddpm_mlx_equivalent as mlx_ddpm

    MLX_AVAILABLE = True
except Exception as exc:
    MLX_IMPORT_ERROR = exc

print("MLX available:", MLX_AVAILABLE)
if MLX_IMPORT_ERROR is not None:
    print("MLX import error:", repr(MLX_IMPORT_ERROR))


## Job Matrix

Each job has one backend, one model family, one beta schedule, and optionally one guided mode. The Torch jobs use the production trainers; the MLX jobs use the equivalent MLX U-Net models and a compact notebook training loop.

In [ ]:
SCHEDULE_ALIAS = {
    "linear": "linear",
    "hash-approach1": "approach1",
    "hash-approach2": "approach2",
}


def jsonable(value):
    if is_dataclass(value):
        value = asdict(value)
    if isinstance(value, Path):
        return str(value)
    if isinstance(value, dict):
        return {str(k): jsonable(v) for k, v in value.items()}
    if isinstance(value, (list, tuple)):
        return [jsonable(v) for v in value]
    return value


def resolve_repo_path(path_value):
    path = Path(path_value)
    return path if path.is_absolute() else REPO_ROOT / path


def normalize_config_paths(config):
    """Force config paths to repo-root absolute paths before schedule/dataset use."""
    updates = {}
    for attr in ("data_root", "json_root", "output_dir", "beta_values_path"):
        if hasattr(config, attr):
            value = getattr(config, attr)
            if value is not None:
                updates[attr] = resolve_repo_path(value)
    if not updates:
        return config
    return type(config)(**{**asdict(config), **updates})


def show_config(config):
    pprint(jsonable(config))


def show_result(result):
    pprint(jsonable(result))


def model_variant_name(family: str, guided_mode: str | None = None) -> str:
    if family == "guided":
        suffix = "cfg" if guided_mode == "classifier-free" else "cls"
        return f"guided_{suffix}"
    return {"base": "base", "loop": "loop", "unconditional": "uncond"}[family]


def output_dir_for(backend: str, family: str, schedule: str, guided_mode: str | None = None) -> Path:
    variant = model_variant_name(family, guided_mode)
    return NOTEBOOK_OUTPUT / backend / f"{variant}_{SCHEDULE_ALIAS[schedule]}"


def common_kwargs(backend: str, family: str, schedule: str, guided_mode: str | None = None) -> dict:
    return dict(
        data_root=DATA_ROOT,
        json_root=JSON_ROOT,
        output_dir=output_dir_for(backend, family, schedule, guided_mode),
        image_size=IMAGE_SIZE,
        channels=CHANNELS,
        fit_mode=FIT_MODE,
        max_images=MAX_IMAGES,
        batch_size=BATCH_SIZE,
        train_steps=TRAIN_STEPS,
        epochs=EPOCHS,
        timesteps=TIMESTEPS,
        learning_rate=LEARNING_RATE,
        base_channels=BASE_CHANNELS,
        time_dim=TIME_DIM,
        beta_schedule=schedule,
        device=TORCH_DEVICE,
        seed=SEED,
        log_every=LOG_EVERY,
        sample_every=SAMPLE_EVERY,
        checkpoint_every=CHECKPOINT_EVERY,
        sample_count=SAMPLE_COUNT,
        save_process_traces=SAVE_PROCESS_TRACES,
        trace_sample_count=TRACE_SAMPLE_COUNT,
    )


def build_config(backend: str, family: str, schedule: str, guided_mode: str | None = None):
    kwargs = common_kwargs(backend, family, schedule, guided_mode)
    if family == "base":
        return ConditionalDiffusionTrainConfig(
            **kwargs,
            label_source="final-hash",
            trace_steps=8,
            save_train_batches_every=0,
            temporal_conditioning="class",
        )
    if family == "guided":
        if guided_mode not in {"classifier-free", "classifier"}:
            raise ValueError(f"Unsupported guided_mode: {guided_mode}")
        return GuidedConditionalDiffusionTrainConfig(
            **kwargs,
            label_source="final-hash",
            trace_steps=8,
            save_train_batches_every=0,
            temporal_conditioning="class",
            guidance_mode=guided_mode,
            guidance_scale=2.0 if guided_mode == "classifier-free" else 1.0,
            condition_dropout=0.1 if guided_mode == "classifier-free" else 0.0,
            classifier_base_channels=BASE_CHANNELS,
            classifier_learning_rate=LEARNING_RATE,
        )
    if family == "loop":
        return LoopConditionedDiffusionTrainConfig(
            **kwargs,
            trace_steps=8,
            save_train_batches_every=0,
            condition_step="4th Step",
            condition_round="1st Round",
            loop_count=64,
        )
    if family == "unconditional":
        kwargs.pop("json_root")
        return UnconditionalDDPMTrainConfig(
            **kwargs,
            save_train_batches_every=0,
        )
    raise ValueError(f"Unsupported family: {family}")


def iter_job_specs() -> list[dict]:
    specs = []
    for backend in SELECTED_BACKENDS:
        for family in SELECTED_MODEL_FAMILIES:
            guided_modes: Iterable[str | None]
            guided_modes = GUIDED_MODES if family == "guided" else (None,)
            for guided_mode in guided_modes:
                for schedule in SELECTED_BETA_SCHEDULES:
                    name = f"{backend}_{model_variant_name(family, guided_mode)}_{SCHEDULE_ALIAS[schedule]}"
                    specs.append(
                        dict(
                            name=name,
                            backend=backend,
                            family=family,
                            guided_mode=guided_mode,
                            schedule=schedule,
                            config=build_config(backend, family, schedule, guided_mode),
                        )
                    )
    return specs


job_specs = iter_job_specs()
print("job count:", len(job_specs))
for idx, spec in enumerate(job_specs):
    print(f"[{idx:02d}] {spec['name']} -> {spec['config'].output_dir}")


## Torch Runner

Torch jobs are delegated to the existing training functions. For `linear`, `timesteps="auto"` is used so its length matches the hash-derived approach schedules.

In [ ]:
def run_torch_job(spec: dict) -> dict:
    config = normalize_config_paths(spec["config"])
    family = spec["family"]
    print(f"Running Torch job: {spec['name']}")
    show_config(config)

    if family == "base":
        result = train_conditional_diffusion(config)
    elif family == "guided":
        result = train_guided_conditional_diffusion(config)
    elif family == "loop":
        result = train_loop_conditioned_diffusion(config)
    elif family == "unconditional":
        result = train_unconditional_ddpm(config)
    else:
        raise ValueError(f"Unsupported family: {family}")

    cleanup_torch_resources()
    return {"job": spec["name"], "backend": "torch", **dict(result)}


## MLX Equivalent Runner

MLX jobs reuse the same datasets and schedules, convert training batches from Torch `NCHW` to MLX `NHWC`, and train the equivalent MLX U-Net model family.

In [ ]:
def torch_nchw_to_mlx_nhwc(images: torch.Tensor):
    return mx.array(images.detach().cpu().permute(0, 2, 3, 1).numpy(), dtype=mx.float32)


def torch_tensor_to_mlx(array: torch.Tensor, *, dtype=None):
    np_array = array.detach().cpu().numpy()
    return mx.array(np_array, dtype=dtype) if dtype is not None else mx.array(np_array)


def mlx_nhwc_to_torch_nchw(samples) -> torch.Tensor:
    mx.eval(samples)
    array = np.asarray(samples, dtype=np.float32)
    return torch.from_numpy(array).permute(0, 3, 1, 2).contiguous()


def scheduler_from_config(config):
    custom_betas = build_beta_schedule(config)
    if isinstance(config.timesteps, int):
        scheduler_timesteps = int(config.timesteps)
    elif custom_betas is not None:
        scheduler_timesteps = int(custom_betas.size)
    else:
        raise ValueError("timesteps='auto' requires a resolvable beta schedule")
    return mlx_ddpm.MLXImageDDPMScheduler(
        timesteps=scheduler_timesteps,
        beta_start=config.beta_start,
        beta_end=config.beta_end,
        betas=custom_betas,
    )


def image_shape_from_dataset(dataset, config) -> tuple[int, int, int]:
    image = dataset[0][0].unsqueeze(0)
    if config.fit_mode != "height-flatten":
        image = _ensure_square_batch(image)
    _, channels, height, width = image.shape
    return int(height), int(width), int(channels)


def make_mlx_dataset(family: str, config):
    if family in {"base", "guided"}:
        return GeneratedImageDataset(
            config.data_root,
            json_root=config.json_root,
            image_size=config.image_size,
            channels=config.channels,
            fit_mode=config.fit_mode,
            condition_mode=config.condition_mode,
            label_source=config.label_source,
            max_images=config.max_images,
            use_loop_images=config.use_loop_images,
            max_loop_count=config.max_loop_count,
        )
    if family == "loop":
        return LoopConditionedImageDataset(
            config.data_root,
            json_root=config.json_root,
            image_size=config.image_size,
            channels=config.channels,
            fit_mode=config.fit_mode,
            condition_step=config.condition_step,
            condition_round=config.condition_round,
            loop_count=config.loop_count,
            word_names=config.word_names,
            max_images=config.max_images,
        )
    if family == "unconditional":
        return UnconditionalImageDataset(
            config.data_root,
            image_size=config.image_size,
            channels=config.channels,
            fit_mode=config.fit_mode,
            max_images=config.max_images,
        )
    raise ValueError(f"Unsupported family: {family}")


def make_mlx_model_and_steps(family: str, guided_mode: str | None, dataset, config, scheduler):
    optimizer = None
    classifier = None
    classifier_step = None

    if family == "base":
        model = mlx_ddpm.MLXConditionalUNet(
            in_channels=dataset.channels,
            num_conditions=dataset.num_conditions,
            base_channels=config.base_channels,
            time_dim=config.time_dim,
            temporal_conditioning=config.temporal_conditioning,
            max_loop_count=config.max_loop_count,
        )
        optimizer = mlx_optim.AdamW(learning_rate=config.learning_rate)
        train_step = mlx_ddpm.make_conditional_train_step(model, scheduler, optimizer)
    elif family == "guided":
        if guided_mode == "classifier-free":
            model_num_conditions = dataset.num_conditions + 1
            model = mlx_ddpm.MLXConditionalUNet(
                in_channels=dataset.channels,
                num_conditions=model_num_conditions,
                base_channels=config.base_channels,
                time_dim=config.time_dim,
                temporal_conditioning=config.temporal_conditioning,
                max_loop_count=config.max_loop_count,
            )
            optimizer = mlx_optim.AdamW(learning_rate=config.learning_rate)
            train_step = mlx_ddpm.make_classifier_free_train_step(
                model,
                scheduler,
                optimizer,
                null_label=dataset.num_conditions,
                condition_dropout=config.condition_dropout,
            )
        elif guided_mode == "classifier":
            model = mlx_ddpm.MLXConditionalUNet(
                in_channels=dataset.channels,
                num_conditions=1,
                base_channels=config.base_channels,
                time_dim=config.time_dim,
                temporal_conditioning=config.temporal_conditioning,
                max_loop_count=config.max_loop_count,
            )
            optimizer = mlx_optim.AdamW(learning_rate=config.learning_rate)
            train_step = mlx_ddpm.make_classifier_guided_denoiser_train_step(
                model,
                scheduler,
                optimizer,
            )
            classifier = mlx_ddpm.MLXNoisyImageClassifier(
                in_channels=dataset.channels,
                num_conditions=dataset.num_conditions,
                base_channels=config.classifier_base_channels,
                time_dim=config.time_dim,
            )
            classifier_optimizer = mlx_optim.AdamW(learning_rate=config.classifier_learning_rate)
            classifier_step = mlx_ddpm.make_noisy_classifier_train_step(
                classifier,
                scheduler,
                classifier_optimizer,
            )
        else:
            raise ValueError(f"Unsupported guided_mode: {guided_mode}")
    elif family == "loop":
        model = mlx_ddpm.MLXLoopConditionedUNet(
            in_channels=dataset.channels,
            condition_shape=dataset.condition_shape,
            base_channels=config.base_channels,
            time_dim=config.time_dim,
            diffusion_timesteps=scheduler.timesteps,
        )
        optimizer = mlx_optim.AdamW(learning_rate=config.learning_rate)
        train_step = mlx_ddpm.make_loop_conditioned_train_step(model, scheduler, optimizer)
    elif family == "unconditional":
        model = mlx_ddpm.MLXUnconditionalUNet(
            in_channels=dataset.channels,
            base_channels=config.base_channels,
            time_dim=config.time_dim,
        )
        optimizer = mlx_optim.AdamW(learning_rate=config.learning_rate)
        train_step = mlx_ddpm.make_unconditional_train_step(model, scheduler, optimizer)
    else:
        raise ValueError(f"Unsupported family: {family}")

    mx.eval(model.parameters())
    if classifier is not None:
        mx.eval(classifier.parameters())
    return model, train_step, classifier, classifier_step


def first_source_image_by_label(dataset, labels: torch.Tensor, *, fit_mode: str) -> torch.Tensor:
    first_by_label = {}
    for index, sample in enumerate(dataset.samples):
        first_by_label.setdefault(int(sample.label), index)

    images = []
    for label in labels.detach().cpu().tolist():
        label_int = int(label)
        if label_int not in first_by_label:
            raise ValueError(f"Label {label_int} not found in dataset")
        images.append(dataset[first_by_label[label_int]][0])
    image_batch = torch.stack(images)
    if fit_mode != "height-flatten":
        image_batch = _ensure_square_batch(image_batch)
    return image_batch


def source_images_for_indices(dataset, indices: torch.Tensor, *, fit_mode: str) -> torch.Tensor:
    images = [dataset[int(index)][0] for index in indices.detach().cpu().tolist()]
    image_batch = torch.stack(images)
    if fit_mode != "height-flatten":
        image_batch = _ensure_square_batch(image_batch)
    return image_batch


def source_condition_names_for_dataset(dataset) -> list[str]:
    return [getattr(sample, "run_id", f"sample_{idx}") for idx, sample in enumerate(dataset.samples)]


def cleanup_legacy_mlx_samples_dir(output_dir: Path) -> None:
    legacy_dir = output_dir / "samples"
    if not legacy_dir.is_dir():
        return

    legacy_patterns = (
        "final.png",
        "final.labels.json",
        "preview.png",
        "preview.labels.json",
        "sample*.png",
        "sample*.json",
    )
    for pattern in legacy_patterns:
        for legacy_path in legacy_dir.glob(pattern):
            if legacy_path.is_file():
                legacy_path.unlink()
    try:
        legacy_dir.rmdir()
    except OSError:
        pass


def save_mlx_trace_state(
    state,
    labels: torch.Tensor,
    condition_names: list[str],
    output_dir: Path,
    filename: str,
) -> list[Path]:
    png_dir = output_dir / "png"
    json_dir = output_dir / "json"
    png_dir.mkdir(parents=True, exist_ok=True)
    json_dir.mkdir(parents=True, exist_ok=True)

    clipped = mx.clip(state, -1.0, 1.0)
    images = mlx_nhwc_to_torch_nchw(clipped)
    result = save_image_grid(
        images,
        labels.detach().cpu(),
        condition_names,
        png_dir / filename,
        json_path=json_dir / filename.replace(".png", ".labels.json"),
    )
    return result["files"]


def mlx_trace_inputs(spec: dict, dataset, config):
    family = spec["family"]
    sample_count = min(config.trace_sample_count, len(dataset))
    if sample_count <= 0:
        raise ValueError("trace_sample_count must be positive")

    if family in {"base", "guided"}:
        images = []
        labels = []
        loop_metas = []
        for index in range(sample_count):
            image, label, loop_meta = dataset[index]
            images.append(image)
            labels.append(int(label))
            loop_metas.append(loop_meta)
        image_batch = torch.stack(images)
        if config.fit_mode != "height-flatten":
            image_batch = _ensure_square_batch(image_batch)
        label_batch = torch.tensor(labels, dtype=torch.long)
        return {
            "x0": torch_nchw_to_mlx_nhwc(image_batch),
            "labels": label_batch,
            "labels_mx": torch_tensor_to_mlx(label_batch, dtype=mx.int32),
            "condition_names": dataset.condition_names,
            "loop_meta_mx": torch_tensor_to_mlx(torch.stack(loop_metas), dtype=mx.float32),
            "conditions_mx": None,
        }

    if family == "loop":
        images = []
        conditions = []
        labels = []
        for index in range(sample_count):
            image, condition, _sample_index = dataset[index]
            images.append(image)
            conditions.append(condition)
            labels.append(index)
        image_batch = torch.stack(images)
        if config.fit_mode != "height-flatten":
            image_batch = _ensure_square_batch(image_batch)
        label_batch = torch.tensor(labels, dtype=torch.long)
        return {
            "x0": torch_nchw_to_mlx_nhwc(image_batch),
            "labels": label_batch,
            "labels_mx": None,
            "condition_names": source_condition_names_for_dataset(dataset),
            "loop_meta_mx": None,
            "conditions_mx": torch_tensor_to_mlx(torch.stack(conditions), dtype=mx.float32),
        }

    if family == "unconditional":
        images = []
        labels = []
        for index in range(sample_count):
            image, _sample_index = dataset[index]
            images.append(image)
            labels.append(index)
        image_batch = torch.stack(images)
        if config.fit_mode != "height-flatten":
            image_batch = _ensure_square_batch(image_batch)
        label_batch = torch.tensor(labels, dtype=torch.long)
        return {
            "x0": torch_nchw_to_mlx_nhwc(image_batch),
            "labels": label_batch,
            "labels_mx": None,
            "condition_names": source_condition_names_for_dataset(dataset),
            "loop_meta_mx": None,
            "conditions_mx": None,
        }

    raise ValueError(f"Unsupported trace family: {family}")


def save_mlx_forward_process_trace(scheduler, x0, labels, condition_names, output_dir: Path):
    saved_paths = []
    print(f"[forward-trace] saving x0 + {scheduler.timesteps} noising steps to {output_dir}")
    saved_paths.extend(save_mlx_trace_state(x0, labels, condition_names, output_dir, "x0.png"))

    noise = mx.random.normal(x0.shape, dtype=mx.float32)
    for step in range(scheduler.timesteps):
        timesteps = mx.full((x0.shape[0],), step, dtype=mx.int32)
        noised = scheduler.q_sample(x0, timesteps, noise)
        filename = f"t_{step:06d}.png"
        saved_paths.extend(save_mlx_trace_state(noised, labels, condition_names, output_dir, filename))
        print(f"[forward-trace] step={step:06d} path={output_dir / 'png' / filename}")
    print(f"[forward-trace] completed: {len(saved_paths)} files")
    return saved_paths


def mlx_reverse_step(spec: dict, model, classifier, scheduler, x, step: int, trace_inputs, config):
    family = spec["family"]
    guided_mode = spec["guided_mode"]
    if family == "base":
        return scheduler.p_sample_conditional(
            model,
            x,
            step,
            trace_inputs["labels_mx"],
            trace_inputs["loop_meta_mx"],
        )
    if family == "guided" and guided_mode == "classifier-free":
        return mlx_ddpm.p_sample_classifier_free_guidance(
            scheduler,
            model,
            x,
            step,
            trace_inputs["labels_mx"],
            loop_meta=trace_inputs["loop_meta_mx"],
            null_label=len(trace_inputs["condition_names"]),
            guidance_scale=config.guidance_scale,
        )
    if family == "guided" and guided_mode == "classifier":
        return mlx_ddpm.p_sample_classifier_guidance(
            scheduler,
            model,
            classifier,
            x,
            step,
            trace_inputs["labels_mx"],
            loop_meta=trace_inputs["loop_meta_mx"],
            guidance_scale=config.guidance_scale,
        )
    if family == "loop":
        return scheduler.p_sample_loop_conditioned(
            model,
            x,
            step,
            trace_inputs["conditions_mx"],
        )
    if family == "unconditional":
        return scheduler.p_sample_unconditional(model, x, step)
    raise ValueError(f"Unsupported reverse trace family: {family}")


def save_mlx_reverse_process_trace(
    spec: dict,
    model,
    classifier,
    scheduler,
    trace_inputs,
    image_shape,
    config,
    output_dir: Path,
):
    labels = trace_inputs["labels"]
    condition_names = trace_inputs["condition_names"]
    sample_count = int(labels.shape[0])
    x = mx.random.normal((sample_count, *image_shape), dtype=mx.float32)

    saved_paths = []
    print(f"[reverse-trace] saving xT + {scheduler.timesteps} denoising steps to {output_dir}")
    saved_paths.extend(save_mlx_trace_state(x, labels, condition_names, output_dir, "xT_noise.png"))
    print(f"[reverse-trace] saved xT: {output_dir / 'png' / 'xT_noise.png'}")

    for step in reversed(range(scheduler.timesteps)):
        x = mlx_reverse_step(spec, model, classifier, scheduler, x, step, trace_inputs, config)
        filename = f"t_{step:06d}.png"
        saved_paths.extend(save_mlx_trace_state(x, labels, condition_names, output_dir, filename))
        print(f"[reverse-trace] step={step:06d} path={output_dir / 'png' / filename}")
    print(f"[reverse-trace] completed: {len(saved_paths)} files")
    return saved_paths


def save_mlx_process_traces(spec: dict, model, classifier, scheduler, dataset, config, image_shape):
    trace_inputs = mlx_trace_inputs(spec, dataset, config)
    trace_dir = config.output_dir / "process_traces"
    forward_paths = save_mlx_forward_process_trace(
        scheduler,
        trace_inputs["x0"],
        trace_inputs["labels"],
        trace_inputs["condition_names"],
        trace_dir / "forward",
    )
    reverse_paths = save_mlx_reverse_process_trace(
        spec,
        model,
        classifier,
        scheduler,
        trace_inputs,
        image_shape,
        config,
        trace_dir / "reverse",
    )
    return {"forward": forward_paths, "reverse": reverse_paths}


def save_mlx_sample(spec: dict, model, classifier, scheduler, dataset, config, image_shape):
    if not SAVE_MLX_SAMPLES:
        return None

    cleanup_legacy_mlx_samples_dir(config.output_dir)

    family = spec["family"]
    guided_mode = spec["guided_mode"]
    sample_count = min(config.sample_count, len(dataset))

    if family == "base":
        labels = mx.array([idx % dataset.num_conditions for idx in range(sample_count)], dtype=mx.int32)
        samples = mlx_ddpm.sample_conditional(scheduler, model, labels, image_shape)
        torch_labels = torch.tensor(np.asarray(labels), dtype=torch.long)
        condition_names = dataset.condition_names
        source_images = first_source_image_by_label(dataset, torch_labels, fit_mode=config.fit_mode)
    elif family == "guided" and guided_mode == "classifier-free":
        labels = mx.array([idx % dataset.num_conditions for idx in range(sample_count)], dtype=mx.int32)
        samples = mlx_ddpm.sample_classifier_free_guidance(
            scheduler,
            model,
            labels,
            image_shape,
            null_label=dataset.num_conditions,
            guidance_scale=config.guidance_scale,
        )
        torch_labels = torch.tensor(np.asarray(labels), dtype=torch.long)
        condition_names = dataset.condition_names
        source_images = first_source_image_by_label(dataset, torch_labels, fit_mode=config.fit_mode)
    elif family == "guided" and guided_mode == "classifier":
        labels = mx.array([idx % dataset.num_conditions for idx in range(sample_count)], dtype=mx.int32)
        samples = mlx_ddpm.sample_classifier_guidance(
            scheduler,
            model,
            classifier,
            labels,
            image_shape,
            guidance_scale=config.guidance_scale,
        )
        torch_labels = torch.tensor(np.asarray(labels), dtype=torch.long)
        condition_names = dataset.condition_names
        source_images = first_source_image_by_label(dataset, torch_labels, fit_mode=config.fit_mode)
    elif family == "loop":
        source_indices = torch.tensor([idx % len(dataset) for idx in range(sample_count)], dtype=torch.long)
        condition_tensors = [dataset[int(idx)][1] for idx in source_indices.tolist()]
        conditions = torch_tensor_to_mlx(torch.stack(condition_tensors), dtype=mx.float32)
        samples = mlx_ddpm.sample_loop_conditioned(scheduler, model, conditions, image_shape)
        torch_labels = source_indices
        condition_names = source_condition_names_for_dataset(dataset)
        source_images = source_images_for_indices(dataset, source_indices, fit_mode=config.fit_mode)
    elif family == "unconditional":
        source_indices = torch.tensor([idx % len(dataset) for idx in range(sample_count)], dtype=torch.long)
        samples = mlx_ddpm.sample_unconditional(scheduler, model, (sample_count, *image_shape))
        torch_labels = source_indices
        condition_names = source_condition_names_for_dataset(dataset)
        source_images = source_images_for_indices(dataset, source_indices, fit_mode=config.fit_mode)
    else:
        raise ValueError(f"Unsupported sample family: {family}")

    final_images = mlx_nhwc_to_torch_nchw(samples)
    sample_paths = save_sample_artifacts(
        source_images,
        final_images,
        torch_labels,
        condition_names,
        config.output_dir / "sample",
    )
    return sample_paths


def write_mlx_metadata(spec: dict, config, scheduler, final_loss: float, classifier_loss: float | None):
    config.output_dir.mkdir(parents=True, exist_ok=True)
    (config.output_dir / "run_config.json").write_text(
        json.dumps(jsonable(config), indent=2, sort_keys=True),
        encoding="utf-8",
    )
    mx.eval(scheduler.betas)
    beta_payload = {
        "backend": "mlx",
        "family": spec["family"],
        "guided_mode": spec["guided_mode"],
        "schedule": spec["schedule"],
        "timesteps": scheduler.timesteps,
        "betas": np.asarray(scheduler.betas).reshape(-1).tolist(),
        "final_loss": final_loss,
        "classifier_loss": classifier_loss,
    }
    (config.output_dir / "beta_schedule.json").write_text(
        json.dumps(beta_payload, indent=2),
        encoding="utf-8",
    )


def cleanup_mlx():
    gc.collect()
    if MLX_AVAILABLE and hasattr(mx, "clear_cache"):
        mx.clear_cache()


In [ ]:
def run_mlx_job(spec: dict) -> dict:
    if not MLX_AVAILABLE:
        raise RuntimeError(f"MLX is unavailable: {MLX_IMPORT_ERROR!r}")

    config = normalize_config_paths(spec["config"])
    family = spec["family"]
    guided_mode = spec["guided_mode"]
    print(f"Running MLX job: {spec['name']}")
    show_config(config)

    set_seed(config.seed)
    mx.random.seed(config.seed)
    dataset = make_mlx_dataset(family, config)
    image_shape = image_shape_from_dataset(dataset, config)
    scheduler = scheduler_from_config(config)
    model, train_step, classifier, classifier_step = make_mlx_model_and_steps(
        family,
        guided_mode,
        dataset,
        config,
        scheduler,
    )

    dataloader = DataLoader(
        dataset,
        batch_size=config.batch_size,
        shuffle=True,
        num_workers=config.num_workers,
        drop_last=False,
    )
    loader_iter = cycle(dataloader)
    effective_steps = resolve_train_steps(
        dataset_size=len(dataset),
        batch_size=config.batch_size,
        train_steps=config.train_steps,
        epochs=config.epochs,
    )

    last_loss = math.nan
    last_classifier_loss = None
    for step in range(1, effective_steps + 1):
        batch = next(loader_iter)
        if family in {"base", "guided"}:
            images, labels, loop_meta = batch
            if config.fit_mode != "height-flatten":
                images = _ensure_square_batch(images)
            x0 = torch_nchw_to_mlx_nhwc(images)
            labels_mx = torch_tensor_to_mlx(labels, dtype=mx.int32)
            loop_meta_mx = torch_tensor_to_mlx(loop_meta, dtype=mx.float32)
            loss = train_step(x0, labels_mx, loop_meta_mx)
            if classifier_step is not None:
                classifier_loss = classifier_step(x0, labels_mx)
                mx.eval(classifier_loss)
                last_classifier_loss = float(classifier_loss)
        elif family == "loop":
            images, conditions, _indices = batch
            if config.fit_mode != "height-flatten":
                images = _ensure_square_batch(images)
            x0 = torch_nchw_to_mlx_nhwc(images)
            conditions_mx = torch_tensor_to_mlx(conditions, dtype=mx.float32)
            loss = train_step(x0, conditions_mx)
        elif family == "unconditional":
            images, _indices = batch
            if config.fit_mode != "height-flatten":
                images = _ensure_square_batch(images)
            x0 = torch_nchw_to_mlx_nhwc(images)
            loss = train_step(x0)
        else:
            raise ValueError(f"Unsupported family: {family}")

        mx.eval(loss)
        last_loss = float(loss)
        if step == 1 or step % config.log_every == 0 or step == effective_steps:
            msg = f"step={step:06d} loss={last_loss:.6f}"
            if last_classifier_loss is not None:
                msg += f" classifier_loss={last_classifier_loss:.6f}"
            print(msg)

    sample_paths = save_mlx_sample(spec, model, classifier, scheduler, dataset, config, image_shape)
    process_trace_paths = None
    if config.save_process_traces:
        process_trace_paths = save_mlx_process_traces(
            spec,
            model,
            classifier,
            scheduler,
            dataset,
            config,
            image_shape,
        )
        print(f"saved process traces: {config.output_dir / 'process_traces'}")
    write_mlx_metadata(spec, config, scheduler, last_loss, last_classifier_loss)
    cleanup_mlx()
    return {
        "job": spec["name"],
        "backend": "mlx",
        "family": family,
        "guided_mode": guided_mode,
        "beta_schedule": spec["schedule"],
        "timesteps": scheduler.timesteps,
        "final_loss": last_loss,
        "classifier_loss": last_classifier_loss,
        "sample_grid": None if sample_paths is None else sample_paths["final"],
        "sample_source_grid": None if sample_paths is None else sample_paths["source"],
        "sample_final_manifest": None if sample_paths is None else sample_paths["final"],
        "sample_source_manifest": None if sample_paths is None else sample_paths["source"],
        "sample_dir": None if sample_paths is None else config.output_dir / "sample",
        "sample_final_dir": None if sample_paths is None else sample_paths["final_dir"],
        "sample_source_dir": None if sample_paths is None else sample_paths["source_dir"],
        "sample_decode_comparison": None if sample_paths is None else sample_paths["decode_comparison"],
        "sample_final_files": None if sample_paths is None else sample_paths["final_files"],
        "sample_source_files": None if sample_paths is None else sample_paths["source_files"],
        "process_traces": (
            None if process_trace_paths is None else config.output_dir / "process_traces"
        ),
        "output_dir": config.output_dir,
    }


def run_job(spec: dict) -> dict:
    if spec["backend"] == "torch":
        return run_torch_job(spec)
    if spec["backend"] == "mlx":
        return run_mlx_job(spec)
    raise ValueError(f"Unsupported backend: {spec['backend']}")


## Job Cells

Each code cell below represents exactly one job. The cell only runs when its job name is enabled by `RUN_FLAGS`; otherwise it prints a skip message. Edit `ENABLED_JOB_NAMES` or `RUN_ALL_JOB_FLAGS` in the setup cell below, then run the job cells you want.


In [ ]:
results = {}
JOB_BY_NAME = {spec["name"]: spec for spec in job_specs}

# -----------------------------------------------------------------------------
# Per-job run flags
# -----------------------------------------------------------------------------
# Keep this False for selective execution. Set True only when you want every job
# cell that is present in JOB_BY_NAME to run.
RUN_ALL_JOB_FLAGS = True

# Put the exact job names to run here. All other job cells will skip.
# Examples:
# ENABLED_JOB_NAMES = ("torch_base_linear", "mlx_base_linear")
# ENABLED_JOB_NAMES = ("torch_loop_approach1",)
ENABLED_JOB_NAMES = (
    # "torch_base_linear",
    # "mlx_base_linear",
)

RUN_FLAGS = {
    name: bool(RUN_ALL_JOB_FLAGS or name in ENABLED_JOB_NAMES)
    for name in JOB_BY_NAME
}


def enabled_job_names() -> list[str]:
    return [name for name, enabled in RUN_FLAGS.items() if enabled]


def show_run_flags() -> None:
    enabled = enabled_job_names()
    print(f"enabled jobs: {len(enabled)} / {len(RUN_FLAGS)}")
    for name in enabled:
        print("  RUN", name)
    if not enabled:
        print("  none; edit ENABLED_JOB_NAMES or set RUN_ALL_JOB_FLAGS=True")


show_run_flags()


def run_named_job(name: str) -> dict:
    if name not in JOB_BY_NAME:
        available = ", ".join(sorted(JOB_BY_NAME))
        raise KeyError(f"Unknown or filtered-out job: {name}. Available jobs: {available}")
    print(f"Running job: {name}")
    result = run_job(JOB_BY_NAME[name])
    results[name] = result
    show_result(result)
    return result


def run_or_skip_job(name: str):
    if name not in JOB_BY_NAME:
        print(f"Skipped {name}: not present in current SELECTED_* filters.")
        return None
    if not RUN_FLAGS.get(name, False):
        print(f"Skipped {name}: RUN_FLAGS[{name!r}] is False.")
        return None
    return run_named_job(name)


def run_enabled_jobs() -> dict:
    for name in enabled_job_names():
        run_named_job(name)
    return results


### TORCH Jobs

Run these cells one at a time when you want explicit per-job execution.

#### torch_base_linear

`torch / base / linear`

In [ ]:
# Job: torch_base_linear
torch_base_linear_result = run_or_skip_job("torch_base_linear")


#### torch_base_approach1

`torch / base / hash-approach1`

In [ ]:
# Job: torch_base_approach1
torch_base_approach1_result = run_or_skip_job("torch_base_approach1")


#### torch_base_approach2

`torch / base / hash-approach2`

In [ ]:
# Job: torch_base_approach2
torch_base_approach2_result = run_or_skip_job("torch_base_approach2")


#### torch_guided_cfg_linear

`torch / guided / classifier-free / linear`

In [ ]:
# Job: torch_guided_cfg_linear
torch_guided_cfg_linear_result = run_or_skip_job("torch_guided_cfg_linear")


#### torch_guided_cfg_approach1

`torch / guided / classifier-free / hash-approach1`

In [ ]:
# Job: torch_guided_cfg_approach1
torch_guided_cfg_approach1_result = run_or_skip_job("torch_guided_cfg_approach1")


#### torch_guided_cfg_approach2

`torch / guided / classifier-free / hash-approach2`

In [ ]:
# Job: torch_guided_cfg_approach2
torch_guided_cfg_approach2_result = run_or_skip_job("torch_guided_cfg_approach2")


#### torch_guided_cls_linear

`torch / guided / classifier / linear`

In [ ]:
# Job: torch_guided_cls_linear
torch_guided_cls_linear_result = run_or_skip_job("torch_guided_cls_linear")


#### torch_guided_cls_approach1

`torch / guided / classifier / hash-approach1`

In [ ]:
# Job: torch_guided_cls_approach1
torch_guided_cls_approach1_result = run_or_skip_job("torch_guided_cls_approach1")


#### torch_guided_cls_approach2

`torch / guided / classifier / hash-approach2`

In [ ]:
# Job: torch_guided_cls_approach2
torch_guided_cls_approach2_result = run_or_skip_job("torch_guided_cls_approach2")


#### torch_loop_linear

`torch / loop / linear`

In [ ]:
# Job: torch_loop_linear
torch_loop_linear_result = run_or_skip_job("torch_loop_linear")


#### torch_loop_approach1

`torch / loop / hash-approach1`

In [ ]:
# Job: torch_loop_approach1
torch_loop_approach1_result = run_or_skip_job("torch_loop_approach1")


#### torch_loop_approach2

`torch / loop / hash-approach2`

In [ ]:
# Job: torch_loop_approach2
torch_loop_approach2_result = run_or_skip_job("torch_loop_approach2")


#### torch_uncond_linear

`torch / unconditional / linear`

In [ ]:
# Job: torch_uncond_linear
torch_uncond_linear_result = run_or_skip_job("torch_uncond_linear")


#### torch_uncond_approach1

`torch / unconditional / hash-approach1`

In [ ]:
# Job: torch_uncond_approach1
torch_uncond_approach1_result = run_or_skip_job("torch_uncond_approach1")


#### torch_uncond_approach2

`torch / unconditional / hash-approach2`

In [ ]:
# Job: torch_uncond_approach2
torch_uncond_approach2_result = run_or_skip_job("torch_uncond_approach2")


### MLX Jobs

Run these cells one at a time when you want explicit per-job execution.

#### mlx_base_linear

`mlx / base / linear`

In [ ]:
# Job: mlx_base_linear
mlx_base_linear_result = run_or_skip_job("mlx_base_linear")


#### mlx_base_approach1

`mlx / base / hash-approach1`

In [ ]:
# Job: mlx_base_approach1
mlx_base_approach1_result = run_or_skip_job("mlx_base_approach1")


#### mlx_base_approach2

`mlx / base / hash-approach2`

In [ ]:
# Job: mlx_base_approach2
mlx_base_approach2_result = run_or_skip_job("mlx_base_approach2")


#### mlx_guided_cfg_linear

`mlx / guided / classifier-free / linear`

In [ ]:
# Job: mlx_guided_cfg_linear
mlx_guided_cfg_linear_result = run_or_skip_job("mlx_guided_cfg_linear")


#### mlx_guided_cfg_approach1

`mlx / guided / classifier-free / hash-approach1`

In [ ]:
# Job: mlx_guided_cfg_approach1
mlx_guided_cfg_approach1_result = run_or_skip_job("mlx_guided_cfg_approach1")


#### mlx_guided_cfg_approach2

`mlx / guided / classifier-free / hash-approach2`

In [ ]:
# Job: mlx_guided_cfg_approach2
mlx_guided_cfg_approach2_result = run_or_skip_job("mlx_guided_cfg_approach2")


#### mlx_guided_cls_linear

`mlx / guided / classifier / linear`

In [ ]:
# Job: mlx_guided_cls_linear
mlx_guided_cls_linear_result = run_or_skip_job("mlx_guided_cls_linear")


#### mlx_guided_cls_approach1

`mlx / guided / classifier / hash-approach1`

In [ ]:
# Job: mlx_guided_cls_approach1
mlx_guided_cls_approach1_result = run_or_skip_job("mlx_guided_cls_approach1")


#### mlx_guided_cls_approach2

`mlx / guided / classifier / hash-approach2`

In [ ]:
# Job: mlx_guided_cls_approach2
mlx_guided_cls_approach2_result = run_or_skip_job("mlx_guided_cls_approach2")


#### mlx_loop_linear

`mlx / loop / linear`

In [ ]:
# Job: mlx_loop_linear
mlx_loop_linear_result = run_or_skip_job("mlx_loop_linear")


#### mlx_loop_approach1

`mlx / loop / hash-approach1`

In [ ]:
# Job: mlx_loop_approach1
mlx_loop_approach1_result = run_or_skip_job("mlx_loop_approach1")


#### mlx_loop_approach2

`mlx / loop / hash-approach2`

In [ ]:
# Job: mlx_loop_approach2
mlx_loop_approach2_result = run_or_skip_job("mlx_loop_approach2")


#### mlx_uncond_linear

`mlx / unconditional / linear`

In [ ]:
# Job: mlx_uncond_linear
mlx_uncond_linear_result = run_or_skip_job("mlx_uncond_linear")


#### mlx_uncond_approach1

`mlx / unconditional / hash-approach1`

In [ ]:
# Job: mlx_uncond_approach1
mlx_uncond_approach1_result = run_or_skip_job("mlx_uncond_approach1")


#### mlx_uncond_approach2

`mlx / unconditional / hash-approach2`

In [ ]:
# Job: mlx_uncond_approach2
mlx_uncond_approach2_result = run_or_skip_job("mlx_uncond_approach2")


## Summary

Run this after one or more job cells to write a summary JSON and print the collected results.

In [ ]:
if results:
    summary_path = NOTEBOOK_OUTPUT / "summary.json"
    summary_path.parent.mkdir(parents=True, exist_ok=True)
    summary_path.write_text(
        json.dumps(jsonable(results), indent=2, sort_keys=True),
        encoding="utf-8",
    )
    print("summary:", summary_path)
    summary_decode_path = NOTEBOOK_OUTPUT / "decode_comparison_summary.json"
    write_decode_comparison_summary(results, summary_decode_path)
    print("decode_comparison_summary:", summary_decode_path)

    for name, result in results.items():
        print(
            name,
            "loss=",
            result.get("final_loss"),
            "sample=",
            result.get("sample_grid"),
            "decode=",
            result.get("sample_decode_comparison"),
        )
else:
    print("No results to summarize yet. Run one or more job cells first.")
